In [2]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

ERROR! Session/line number was not unique in database. History logging moved to new session 10


In [3]:
path = '/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/HW1/rt_bahrain/rt_bahrain.edges'
df = pd.read_csv(
    path,
    sep=",",                  # whitespace-separated
    header=None,                 # no header row in file
    names=["retweeter", "original_author", "timestamp"],  # assign names
    engine="python"
)
G_citation = nx.from_pandas_edgelist(df, 'retweeter', 'original_author', 'timestamp', create_using=nx.DiGraph())

print(G_citation.number_of_nodes(), 'nodes', G_citation.number_of_edges(), 'edges')
print(df)

4676 nodes 8007 edges
      retweeter  original_author   timestamp
0          4165              533  1348022857
1          4165             1016  1348026203
2          2445              550  1347990443
3          2445             4177  1347990219
4          2445              201  1347989927
...         ...              ...         ...
8002        298             1140  1347997801
8003       3964             1890  1348017528
8004       3964             3802  1348017551
8005       1549             2595  1348021567
8006       1616             2121  1348059465

[8007 rows x 3 columns]


In [4]:
# For now, remove all isolated nodes from the citation networks. Only keep the connected component
def directed_1in1out_core(H):
    Gc = H.copy()
    # Iteratively prune nodes with in==0 or out==0 (like a directed 1-in/1-out core)
    changed = True
    while changed:
        changed = False
        to_drop = [n for n in Gc if Gc.in_degree(n) == 0 or Gc.out_degree(n) == 0]
        if to_drop:
            Gc.remove_nodes_from(to_drop)
            changed = True
    return Gc

# Build swappable core (keep a mapping to reattach outside degrees later if desired)
G_core = directed_1in1out_core(G_citation)
# print("Original:", G_citation.number_of_nodes(), "nodes,", G_citation.number_of_edges(), "edges")
# print("Core:    ", G_core.number_of_nodes(), "nodes,", G_core.number_of_edges(), "edges")

# If the core is small, directed swaps will still be hard; scale nswap to the actual core size
Ecore = G_core.number_of_edges()

# Try swaps with conservative targets and generous attempts
nswap_target = 3 * Ecore
max_tries    = 20 * Ecore   

G_rand_core = G_core.copy()
nx.directed_edge_swap(G_rand_core, nswap=nswap_target, max_tries=max_tries, seed=42)

# stitch the core back together with untouched exterior nodes:
G_randomized = G_citation.copy()
G_randomized.remove_nodes_from(G_core.nodes)
G_randomized = nx.compose(G_randomized, G_rand_core)  # put randomized core back in


In [5]:
# Clustering coefficient distribution
# Directed local clustering (Fagiolo, used by NetworkX for DiGraph)
c_dir_emp  = nx.clustering(G_citation)       
c_dir_rand = nx.clustering(G_randomized)

# Averages & transitivity (global)
avg_dir_emp   = nx.average_clustering(G_citation)
avg_dir_rand  = nx.average_clustering(G_randomized)
trans_emp     = nx.transitivity(G_citation)       # triangle density (global)
trans_rand    = nx.transitivity(G_randomized)

print(f"Directed average clustering (empirical):  {avg_dir_emp:.4f}")
print(f"Directed average clustering (randomized): {avg_dir_rand:.4f}")
print(f"Transitivity (empirical):                 {trans_emp:.4f}")
print(f"Transitivity (randomized):                {trans_rand:.4f}")
print(avg_dir_emp)

Directed average clustering (empirical):  0.0090
Directed average clustering (randomized): 0.0032
Transitivity (empirical):                 0.0018
Transitivity (randomized):                0.0008
0.009023453173790295


In [6]:
from scipy.stats import spearmanr

# ---------------- Helper for log-binned PDFs ----------------
def log_binned_pdf(x, nbins=50, xmin=None, xmax=None):
    """
    Compute a log-binned probability density for positive data x.
    Returns (x_centers, y_density) suitable for log-log plotting.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    if x.size == 0:
        return np.array([]), np.array([])
    if xmin is None: xmin = x.min()
    if xmax is None: xmax = x.max()
    if xmin == xmax:
        return np.array([xmin]), np.array([1.0])
    edges = np.geomspace(xmin, xmax, nbins + 1)
    counts, edges = np.histogram(x, bins=edges)
    widths  = np.diff(edges)
    centers = np.sqrt(edges[:-1] * edges[1:])  # geometric mean
    pdf = counts / (counts.sum() * widths)
    m = pdf > 0
    return centers[m], pdf[m]

# ---------------- Degree distributions ----------------
nbins = 50
x_in_emp,  y_in_emp  = degree_distribution(G_citation,   number_of_bins=nbins, log_binning=True, directed='in')
x_out_emp, y_out_emp = degree_distribution(G_citation,   number_of_bins=nbins, log_binning=True, directed='out')
x_in_rand, y_in_rand = degree_distribution(G_randomized, number_of_bins=nbins, log_binning=True, directed='in')
x_out_rand,y_out_rand= degree_distribution(G_randomized, number_of_bins=nbins, log_binning=True, directed='out')

# ---------------- Clustering coefficient PDFs ----------------
clust_emp_dict  = nx.clustering(G_citation)      # directed Fagiolo definition
clust_rand_dict = nx.clustering(G_randomized)
clust_emp  = np.fromiter(clust_emp_dict.values(),  dtype=float)
clust_rand = np.fromiter(clust_rand_dict.values(), dtype=float)
c_emp_pos  = clust_emp[clust_emp > 0]
c_rand_pos = clust_rand[clust_rand > 0]
c_xmin = min(c_emp_pos.min(), c_rand_pos.min()) if (c_emp_pos.size and c_rand_pos.size) else None
x_c_emp,  y_c_emp  = log_binned_pdf(c_emp_pos,  nbins=nbins, xmin=c_xmin, xmax=1.0)
x_c_rand, y_c_rand = log_binned_pdf(c_rand_pos, nbins=nbins, xmin=c_xmin, xmax=1.0)

# ---------------- Betweenness centrality PDFs ----------------
bc_emp_dict  = nx.betweenness_centrality(G_citation,   normalized=True, endpoints=False)
bc_rand_dict = nx.betweenness_centrality(G_randomized, normalized=True, endpoints=False)
bc_emp  = np.fromiter(bc_emp_dict.values(),  dtype=float)
bc_rand = np.fromiter(bc_rand_dict.values(), dtype=float)
bc_emp_pos  = bc_emp[bc_emp > 0]
bc_rand_pos = bc_rand[bc_rand > 0]
b_xmin = min(bc_emp_pos.min(), bc_rand_pos.min()) if (bc_emp_pos.size and bc_rand_pos.size) else None
x_b_emp,  y_b_emp  = log_binned_pdf(bc_emp_pos,  nbins=nbins, xmin=b_xmin, xmax=1.0)
x_b_rand, y_b_rand = log_binned_pdf(bc_rand_pos, nbins=nbins, xmin=b_xmin, xmax=1.0)

# ---------------- Plot all three ----------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=150)

# Panel 1: Degree distributions
ax = axes[0]
ax.loglog(x_in_emp,  y_in_emp,  'o', color='teal',           alpha=0.8, mec='.2', label='Empirical in-degree')
ax.loglog(x_out_emp, y_out_emp, 's', color='cornflowerblue', alpha=0.8, mec='.2', label='Empirical out-degree')
ax.loglog(x_in_rand, y_in_rand, 'o', color='firebrick',      alpha=0.8, mec='.2', label='Randomized in-degree')
ax.loglog(x_out_rand,y_out_rand,'s', color='orange',         alpha=0.8, mec='.2', label='Randomized out-degree')
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$P(k)$")
ax.set_title("Degree distribution")
ax.legend(fontsize='x-small', frameon=False)
ax.grid(linewidth=1.2, alpha=0.2)

# Panel 2: Clustering coefficient PDFs
ax = axes[1]
if x_c_emp.size:  ax.loglog(x_c_emp,  y_c_emp,  'o', alpha=0.85, mec='.2', label='Empirical C')
if x_c_rand.size: ax.loglog(x_c_rand, y_c_rand, 's', alpha=0.85, mec='.2', label='Randomized C')
ax.set_xlabel(r"$C$")
ax.set_ylabel(r"$P(C)$")
ax.set_title("Clustering coefficient PDF")
ax.legend(fontsize='x-small', frameon=False)
ax.grid(linewidth=1.2, alpha=0.2)

# Panel 3: Betweenness centrality PDFs
ax = axes[2]
if x_b_emp.size:  ax.loglog(x_b_emp,  y_b_emp,  'o', alpha=0.85, mec='.2', label='Empirical betweenness')
if x_b_rand.size: ax.loglog(x_b_rand, y_b_rand, 's', alpha=0.85, mec='.2', label='Randomized betweenness')
ax.set_xlabel(r"$b$")
ax.set_ylabel(r"$P(b)$")
ax.set_title("Betweenness centrality PDF")
ax.legend(fontsize='x-small', frameon=False)
ax.grid(linewidth=1.2, alpha=0.2)

plt.tight_layout()
plt.show()


NameError: name 'degree_distribution' is not defined